# Project Python Foundations: FoodHub Data Analysis

### Context

The number of restaurants in New York is increasing day by day. Lots of students and busy professionals rely on those restaurants due to their hectic lifestyles. Online food delivery service is a great option for them. It provides them with good food from their favorite restaurants. A food aggregator company FoodHub offers access to multiple restaurants through a single smartphone app.

The app allows the restaurants to receive a direct online order from a customer. The app assigns a delivery person from the company to pick up the order after it is confirmed by the restaurant. The delivery person then uses the map to reach the restaurant and waits for the food package. Once the food package is handed over to the delivery person, he/she confirms the pick-up in the app and travels to the customer's location to deliver the food. The delivery person confirms the drop-off in the app after delivering the food package to the customer. The customer can rate the order in the app. The food aggregator earns money by collecting a fixed margin of the delivery order from the restaurants.

### Objective

The food aggregator company has stored the data of the different orders made by the registered customers in their online portal. They want to analyze the data to get a fair idea about the demand of different restaurants which will help them in enhancing their customer experience. Suppose you are hired as a Data Scientist in this company and the Data Science team has shared some of the key questions that need to be answered. Perform the data analysis to find answers to these questions that will help the company to improve the business.

### Data Description

The data contains the different data related to a food order. The detailed data dictionary is given below.

### Data Dictionary

* order_id: Unique ID of the order
* customer_id: ID of the customer who ordered the food
* restaurant_name: Name of the restaurant
* cuisine_type: Cuisine ordered by the customer
* cost_of_the_order: Cost of the order
* day_of_the_week: Indicates whether the order is placed on a weekday or weekend (The weekday is from Monday to Friday and the weekend is Saturday and Sunday)
* rating: Rating given by the customer out of 5
* food_preparation_time: Time (in minutes) taken by the restaurant to prepare the food. This is calculated by taking the difference between the timestamps of the restaurant's order confirmation and the delivery person's pick-up confirmation.
* delivery_time: Time (in minutes) taken by the delivery person to deliver the food package. This is calculated by taking the difference between the timestamps of the delivery person's pick-up confirmation and drop-off information

### Let us start by importing the required libraries

In [ ]:
# Installing the libraries with the specified version.
!pip install numpy==2.0.2 pandas==2.2.2 matplotlib==3.10.0 seaborn==0.13.2 -q --user

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Import libraries for data manipulation
import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Utility for nicer outputs in notebooks
from IPython.display import display

### Understanding the structure of the data

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Reading the dataset
data_path = Path("foodhub_order.csv")
if not data_path.exists():
    data_path = Path("/mnt/data/foodhub_order.csv")  # fallback (for this environment)

df = pd.read_csv(data_path)
print("Data loaded successfully!")
print("Shape:", df.shape)

In [ ]:
df.head()

### **Question 1:** How many rows and columns are present in the data? [0.5 mark]

In [ ]:
rows, cols = df.shape
print(f"Rows: {rows}, Columns: {cols}")

#### Observations:

- The dataset contains **1898 rows** and **9 columns**.

### **Question 2:** What are the datatypes of the different columns in the dataset? (The info() function can be used) [0.5 mark]

In [ ]:
df.info()

#### Observations:

- `order_id` and `customer_id` are integers.
- `cost_of_the_order` is a float.
- `restaurant_name`, `cuisine_type`, and `day_of_the_week` are categorical (object).
- `rating` is stored as object initially (because of `Not given`) and will be converted to numeric in the next step.
- `food_preparation_time` and `delivery_time` are integers.

### **Question 3:** Are there any missing values in the data? If yes, treat them using an appropriate method. [1 mark]

In [ ]:
# Checking for missing values
print("Missing values (NaN) per column:")
display(df.isna().sum())

# Treating 'Not given' in rating as missing and converting rating to numeric
df["rating"] = pd.to_numeric(df["rating"].replace("Not given", np.nan), errors="coerce")

print("\nAfter treating 'Not given' as NaN:")
display(df.isna().sum())

#### Observations:

- There are **no NaN missing values** in the dataset.
- However, the `rating` column has **736 values as `Not given`**, which are treated as missing by converting them to **NaN**.
- After cleaning, `rating` becomes numeric, which makes it easier to analyze average ratings and rating-based conditions.

### **Question 4:** Check the statistical summary of the data. What is the minimum, average, and maximum time it takes for food to be prepared once an order is placed? [2 marks]

In [ ]:
# Statistical summary
display(df.describe(include="all").T)

# Min, average, max preparation time
prep_stats = df["food_preparation_time"].agg(["min", "mean", "max"])
print("\nFood preparation time (minutes):")
print(f"Min: {prep_stats['min']}")
print(f"Average: {prep_stats['mean']:.2f}")
print(f"Max: {prep_stats['max']}")

#### Observations:

- Food preparation time ranges from **20 min** to **35 min**.
- The **average** food preparation time is **~27.37 min**.

### **Question 5:** How many orders are not rated? [1 mark]

In [ ]:
not_rated = df["rating"].isna().sum()
print("Number of orders not rated:", not_rated)

#### Observations:

- **736 orders** are **not rated** (rating is missing).

### Exploratory Data Analysis (EDA)

### Univariate Analysis

### **Question 6:** Explore all the variables and provide observations on their distributions. (Generally, histograms, boxplots, countplots, etc. are used for univariate exploration.) [9 marks]

In [ ]:
# Univariate analysis (distributions)

# Numerical variables
num_cols = ["cost_of_the_order", "food_preparation_time", "delivery_time"]
for col in num_cols:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()

# Categorical variables
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="day_of_the_week")
plt.title("Orders by day of the week")
plt.show()

plt.figure(figsize=(9, 4))
top_cuisines = df["cuisine_type"].value_counts().head(10).index
sns.countplot(data=df[df["cuisine_type"].isin(top_cuisines)], x="cuisine_type", order=top_cuisines)
plt.xticks(rotation=45, ha="right")
plt.title("Top 10 cuisines by number of orders")
plt.show()

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="rating")
plt.title("Ratings distribution (NaN = not rated)")
plt.show()

print("Unique restaurants:", df["restaurant_name"].nunique())
print("Unique customers:", df["customer_id"].nunique())
print("Unique cuisines:", df["cuisine_type"].nunique())

### **Question 7**: Which are the top 5 restaurants in terms of the number of orders received? [1 mark]

In [ ]:
top5_restaurants = df["restaurant_name"].value_counts().head(5)
display(top5_restaurants)

plt.figure(figsize=(8, 4))
sns.barplot(x=top5_restaurants.values, y=top5_restaurants.index)
plt.title("Top 5 restaurants by number of orders")
plt.xlabel("Number of orders")
plt.ylabel("Restaurant")
plt.show()

#### Observations:

- The top 5 restaurants by number of orders are:
  1. **Shake Shack**
  2. **The Meatball Shop**
  3. **Blue Ribbon Sushi**
  4. **Blue Ribbon Fried Chicken**
  5. **Parm**

### **Question 8**: Which is the most popular cuisine on weekends? [1 mark]

In [ ]:
weekend_orders = df[df["day_of_the_week"] == "Weekend"]
top_weekend_cuisine = weekend_orders["cuisine_type"].value_counts().head(1)
display(top_weekend_cuisine)

#### Observations:

- The most popular cuisine on weekends is **American** cuisine.

### **Question 9**: What percentage of the orders cost more than 20 dollars? [2 marks]

In [ ]:
percentage_over_20 = (df["cost_of_the_order"] > 20).mean() * 100
print(f"Percentage of orders costing more than $20: {percentage_over_20:.2f}%")

#### Observations:

- Approximately **29.24%** of orders cost **more than $20**.

### **Question 10**: What is the mean order delivery time? [1 mark]

In [ ]:
mean_delivery_time = df["delivery_time"].mean()
print(f"Mean order delivery time: {mean_delivery_time:.2f} minutes")

#### Observations:

- The mean delivery time is **~24.16 minutes**.

### **Question 11:** The company has decided to give 20% discount vouchers to the top 3 most frequent customers. Find the IDs of these customers and the number of orders they placed. [1 mark]

In [ ]:
top3_customers = df["customer_id"].value_counts().head(3)
result_q11 = top3_customers.reset_index()
result_q11.columns = ["customer_id", "number_of_orders"]
display(result_q11)

#### Observations:

- The top 3 most frequent customers are **52832 (13 orders)**, **47440 (10 orders)**, and **83287 (9 orders)**.

### Multivariate Analysis

### **Question 12**: Perform a multivariate analysis to explore relationships between the important variables in the dataset. (It is a good idea to explore relations between numerical variables as well as relations between numerical and categorical variables) [10 marks]


In [ ]:
# Multivariate analysis

# Create total_time for analysis
df["total_time"] = df["food_preparation_time"] + df["delivery_time"]

# Correlation heatmap for numerical variables
plt.figure(figsize=(7, 5))
sns.heatmap(
    df[["cost_of_the_order", "food_preparation_time", "delivery_time", "rating", "total_time"]].corr(),
    annot=True,
    fmt=".2f"
)
plt.title("Correlation heatmap (numerical variables)")
plt.show()

# Delivery time by weekday/weekend
plt.figure(figsize=(5, 4))
sns.boxplot(data=df, x="day_of_the_week", y="delivery_time")
plt.title("Delivery time by day of the week")
plt.show()

# Total time by weekday/weekend
plt.figure(figsize=(5, 4))
sns.boxplot(data=df, x="day_of_the_week", y="total_time")
plt.title("Total time (prep + delivery) by day of the week")
plt.show()

# Cost vs total time
plt.figure(figsize=(6, 4))
sns.scatterplot(data=df, x="cost_of_the_order", y="total_time", alpha=0.6)
plt.title("Cost of order vs Total time")
plt.show()

# Average rating by cuisine (only cuisines with enough ratings for stability)
cuisine_summary = (
    df.dropna(subset=["rating"])
      .groupby("cuisine_type")["rating"]
      .agg(["count", "mean"])
      .sort_values("mean", ascending=False)
)
display(cuisine_summary.head(10))

### **Question 13:** The company wants to provide a promotional offer in the advertisement of the restaurants. The condition to get the offer is that the restaurants must have a rating count of more than 50 and the average rating should be greater than 4. Find the restaurants fulfilling the criteria to get the promotional offer. [3 marks]

In [ ]:
# Restaurants with rating count > 50 and average rating > 4
promo_restaurants = (
    df.dropna(subset=["rating"])
      .groupby("restaurant_name")["rating"]
      .agg(["count", "mean"])
      .query("count > 50 and mean > 4")
      .sort_values(["mean", "count"], ascending=[False, False])
)
display(promo_restaurants)

#### Observations:

- The following restaurants satisfy **(rating count > 50)** and **(average rating > 4)**, so they qualify for the promotional offer:
  - **The Meatball Shop**
  - **Blue Ribbon Fried Chicken**
  - **Shake Shack**
  - **Blue Ribbon Sushi**

### **Question 14:** The company charges the restaurant 25% on the orders having cost greater than 20 dollars and 15% on the orders having cost greater than 5 dollars. Find the net revenue generated by the company across all orders. [3 marks]

In [ ]:
# Net revenue: 25% commission if cost > 20, else 15% commission if cost > 5, else 0
commission_rate = np.where(df["cost_of_the_order"] > 20, 0.25,
                           np.where(df["cost_of_the_order"] > 5, 0.15, 0.0))
df["company_revenue"] = df["cost_of_the_order"] * commission_rate

net_revenue = df["company_revenue"].sum()
print(f"Net revenue generated by the company across all orders: ${net_revenue:.2f}")

#### Observations:

- The company's net revenue (based on the given commission rules) is **$6,166.30**.

### **Question 15:** The company wants to analyze the total time required to deliver the food. What percentage of orders take more than 60 minutes to get delivered from the time the order is placed? (The food has to be prepared and then delivered.) [2 marks]

In [ ]:
# Percentage of orders taking more than 60 minutes in total (prep + delivery)
df["total_time"] = df["food_preparation_time"] + df["delivery_time"]
percentage_over_60 = (df["total_time"] > 60).mean() * 100
print(f"Percentage of orders taking more than 60 minutes (total): {percentage_over_60:.2f}%")

#### Observations:

- About **10.54%** of orders take **more than 60 minutes** in total (food prep + delivery).

### **Question 16:** The company wants to analyze the delivery time of the orders on weekdays and weekends. How does the mean delivery time vary during weekdays and weekends? [2 marks]

In [ ]:
mean_delivery_by_day = df.groupby("day_of_the_week")["delivery_time"].mean()
display(mean_delivery_by_day)

diff = mean_delivery_by_day["Weekday"] - mean_delivery_by_day["Weekend"]
print(f"Weekday mean delivery time is higher than weekend by {diff:.2f} minutes.")

#### Observations:

- Mean delivery time is **higher on weekdays (~28.34 min)** compared to weekends (**~22.47 min**), i.e., weekdays take **~5.87 minutes longer** on average.

### Conclusion and Recommendations

### **Question 17:** What are your conclusions from the analysis? What recommendations would you like to share to help improve the business? (You can use cuisine type and feedback ratings to drive your business recommendations.) [6 marks]

### Conclusions:
- Most orders come during the **weekend** (much higher order volume than weekdays).
- **American** and **Japanese** cuisines dominate order volume.
- Delivery time is **significantly higher on weekdays** than weekends, indicating weekday operational/logistics constraints.
- A sizable portion of orders (**~29%**) are high-value (>$20), which are important for revenue.
- Only a few high-volume restaurants meet the promo criteria (high rating count + high average rating).

### Recommendations:
- **Target weekend campaigns** (combo offers / free delivery thresholds) since demand is highest on weekends.
- **Reduce weekday delivery delays** by improving driver availability, route batching, or restaurant pickup coordination on weekdays.
- **Promote high-performing restaurants** (e.g., those meeting the promo criteria) prominently in the app to increase conversion.
- **Expand ratings collection** (e.g., in-app nudges, small reward points) because many orders are unrated, limiting feedback-driven decisions.
- Use cuisine insights: focus marketing on high-demand cuisines (**American/Japanese/Italian/Chinese**) while also highlighting high-rated niche cuisines to diversify demand.

---